## Step 3B — Joint BIO tagger (RoBERTa-base, TherapistQA)

A token-classification head is fine-tuned on top of `roberta-base` over 21 labels
(`O`, `B-<CAT>`, `I-<CAT>` for the 10 distortion categories from Burns).
Inputs are full `patient_question` sequences; outputs are per-token label predictions.

Evaluation uses `seqeval` for token-level F1 and span-level F1 (strict, exact-match).
Category-level F1 is reported by aggregating predicted tag runs into spans and taking
the dominant category per span. The 3C (binary span) scheme is reported alongside as
an upper bound, since it is recoverable from the 3B predictions by collapsing labels.

Inputs (produced by notebook `00_06_bio_tag_therapistQA.ipynb`):
- `processed/therapistqa_bio_3b_{train,val,test}.parquet`
- `processed/therapistqa_bio_label_maps.json`

In [1]:
!pip install -q transformers datasets accelerate evaluate seqeval

from google.colab import drive
drive.mount('/content/drive')

import json
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)
from seqeval.metrics import (
    classification_report as seq_classification_report,
    f1_score as seq_f1,
    precision_score as seq_precision,
    recall_score as seq_recall,
)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device:         {torch.cuda.get_device_name(0)}")
print(f"GPU memory:     {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

DATA_DIR   = Path("/content/drive/MyDrive/thesis_stage2")
MODEL_NAME = "roberta-base"
MAX_LENGTH = 512
SEED       = 42

LABEL_MAPS_FN  = DATA_DIR / "therapistqa_bio_label_maps.json"
PARQUET_TPL = "therapistqa_bio_3c_{split}.parquet"
OUT_DIR = DATA_DIR / "models" / "bio_3c_roberta_base"

OUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
Mounted at /content/drive
CUDA available: True
Device:         Tesla T4
GPU memory:     14.6 GB


In [2]:
with open(LABEL_MAPS_FN, encoding="utf-8") as f:
    label_maps = json.load(f)

LABELS_3B   = label_maps["labels_3c"]
LABEL2ID_3B = label_maps["label2id_3c"]
ID2LABEL_3B = {int(k): v for k, v in label_maps["id2label_3c"].items()}
CATEGORIES  = label_maps["categories"]
NUM_LABELS  = len(LABELS_3B)
print(f"3B labels: {NUM_LABELS}  (O + 2×{len(CATEGORIES)} categories)")

def load_split(split: str) -> pd.DataFrame:
    return pd.read_parquet(DATA_DIR / PARQUET_TPL.format(split=split))

bio_dfs = {s: load_split(s) for s in ("train", "val", "test")}
for s, df in bio_dfs.items():
    print(f"{s}: {len(df)} rows | distorted={df['is_distorted'].sum()}")

3B labels: 3  (O + 2×10 categories)
train: 1768 rows | distorted=1116
val: 379 rows | distorted=239
test: 379 rows | distorted=239


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)


def encode_record(rec):
    """
    Tokenise raw text with offset_mapping, then assign BIO labels to each
    subword based on whether its character range falls inside a gold span.
    """
    enc = tokenizer(
        rec["patient_question"],
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
    )

    offsets = enc.pop("offset_mapping")
    spans = rec["spans"]                           # list of (start, end) char offsets
    category = rec["dominant_distortion"] if rec["is_distorted"] else None

    labels = []
    prev_in_span_idx = None     # index of the active span the previous token belonged to

    for (s, e) in offsets:
        # Special tokens: <s>, </s>, padding -> offset is (0, 0)
        if s == e:
            labels.append(-100)
            prev_in_span_idx = None
            continue

        # Find which gold span (if any) this subword falls into.
        # A subword is "inside" a span if its char range overlaps the span.
        in_span_idx = None
        if category is not None:
            for i, (sp_s, sp_e) in enumerate(spans):
                if s < sp_e and e > sp_s:           # any overlap
                    in_span_idx = i
                    break

        if in_span_idx is None:
            labels.append(LABEL2ID_3B["O"])
        elif in_span_idx != prev_in_span_idx:
            labels.append(LABEL2ID_3B["B-DIST"])
        else:
            labels.append(LABEL2ID_3B["I-DIST"])

        prev_in_span_idx = in_span_idx

    enc["labels"] = labels
    return enc


def to_dataset(df: pd.DataFrame) -> Dataset:
    keep = ["id", "patient_question", "spans", "is_distorted",
            "dominant_distortion", "n_spans"]
    # Convert spans (numpy arrays from parquet) to plain lists of tuples.
    df = df.copy()
    df["spans"] = df["spans"].apply(lambda xs: [tuple(map(int, x)) for x in xs])
    ds = Dataset.from_pandas(df[keep].reset_index(drop=True), preserve_index=False)
    ds = ds.map(encode_record, remove_columns=["patient_question", "spans"])
    return ds


dsd = DatasetDict({s: to_dataset(df) for s, df in bio_dfs.items()})
print(dsd)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1768 [00:00<?, ? examples/s]

Map:   0%|          | 0/379 [00:00<?, ? examples/s]

Map:   0%|          | 0/379 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'is_distorted', 'dominant_distortion', 'n_spans', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1768
    })
    val: Dataset({
        features: ['id', 'is_distorted', 'dominant_distortion', 'n_spans', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 379
    })
    test: Dataset({
        features: ['id', 'is_distorted', 'dominant_distortion', 'n_spans', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 379
    })
})


In [4]:
flat = []
for tags in bio_dfs["train"]["label_ids"]:
    flat.extend(tags)
counts = Counter(flat)

freqs = np.array([counts.get(i, 0) for i in range(NUM_LABELS)], dtype=np.float64)

# Sqrt-inverse-frequency, no mean normalisation: each class gets a weight
# directly proportional to sqrt(max_count / its_count).
raw_weights = np.sqrt(freqs.max() / np.maximum(freqs, 1.0))
class_weights = torch.tensor(np.clip(raw_weights, 1.0, 4.0), dtype=torch.float32)

print("Class weights:")
for i, lab in enumerate(LABELS_3B):
    print(f"  {i:>2} {lab:<28s} count={int(freqs[i]):>7d}  w={class_weights[i].item():.3f}")

Class weights:
   0 O                            count= 307419  w=1.000
   1 B-DIST                       count=   1258  w=4.000
   2 I-DIST                       count=  48362  w=2.521


In [5]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights: torch.Tensor = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            ignore_index=-100,
        )
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [6]:
def ids_to_tags(label_ids_2d, predictions_2d):
    """
    Convert (labels, preds) arrays to two parallel lists of BIO-tag sequences,
    skipping positions where the gold label is -100 (special tokens / padding).
    """
    true_tags, pred_tags = [], []
    for lab_row, pred_row in zip(label_ids_2d, predictions_2d):
        true_seq, pred_seq = [], []
        for li, pi in zip(lab_row, pred_row):
            if li == -100:
                continue
            true_seq.append(ID2LABEL_3B[int(li)])
            pred_seq.append(ID2LABEL_3B[int(pi)])
        true_tags.append(true_seq)
        pred_tags.append(pred_seq)
    return true_tags, pred_tags


def collapse_to_3c(tag_seq):
    """3B -> 3C label projection (B-<X> -> B-DIST, I-<X> -> I-DIST, O -> O)."""
    out = []
    for t in tag_seq:
        if t == "O":
            out.append("O")
        elif t.startswith("B-"):
            out.append("B-DIST")
        else:
            out.append("I-DIST")
    return out


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    true_tags, pred_tags = ids_to_tags(labels, preds)

    # Token-level micro-F1: flatten and compute over labels (excluding O for fair detection F1)
    flat_true = [t for seq in true_tags for t in seq]
    flat_pred = [t for seq in pred_tags for t in seq]
    correct_tokens = sum(t == p for t, p in zip(flat_true, flat_pred))
    token_acc = correct_tokens / max(1, len(flat_true))

    # Span-level (strict, seqeval = exact boundary + label match) on 3B labels
    span_p = seq_precision(true_tags, pred_tags, mode="strict", zero_division=0)
    span_r = seq_recall(true_tags, pred_tags, mode="strict", zero_division=0)
    span_f = seq_f1(true_tags, pred_tags, mode="strict", zero_division=0)

    # Span-level on the 3C projection: detection only, no category
    true_3c = [collapse_to_3c(s) for s in true_tags]
    pred_3c = [collapse_to_3c(s) for s in pred_tags]
    span_f_3c = seq_f1(true_3c, pred_3c, mode="strict", zero_division=0)

    return {
        "token_accuracy": token_acc,
        "span_precision_strict": span_p,
        "span_recall_strict": span_r,
        "span_f1_strict": span_f,
        "span_f1_3c": span_f_3c,
    }

In [7]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL_3B,
    label2id=LABEL2ID_3B,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=12,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="span_f1_strict",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=dsd["train"],
    eval_dataset=dsd["val"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [8]:
train_result = trainer.train()
trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))

print("Train metrics:", train_result.metrics)
print("\nFinal validation metrics:", trainer.evaluate(dsd["val"]))

Epoch,Training Loss,Validation Loss,Token Accuracy,Span Precision Strict,Span Recall Strict,Span F1 Strict,Span F1 3c
1,1.210713,0.636946,0.866439,0.000000,0.000000,0.000000,0.000000
2,1.018222,0.510237,0.851045,0.000971,0.003650,0.001534,0.001534
3,0.860502,0.496184,0.833587,0.041246,0.178832,0.067031,0.067031
4,0.620263,0.584202,0.821563,0.040309,0.171533,0.065278,0.065278
5,0.412800,0.769587,0.842911,0.062663,0.175182,0.092308,0.092308
6,0.257986,0.931087,0.838097,0.063158,0.175182,0.092843,0.092843
7,0.184735,1.229156,0.856189,0.056250,0.098540,0.071618,0.071618
8,0.124232,1.181211,0.852958,0.070140,0.127737,0.090556,0.090556
9,0.102522,1.427202,0.845775,0.074534,0.131387,0.095112,0.095112
10,0.075292,1.614100,0.850475,0.073059,0.116788,0.089888,0.089888


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (7 outputs left out)

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Train metrics: {'train_runtime': 714.0212, 'train_samples_per_second': 29.713, 'train_steps_per_second': 1.865, 'total_flos': 4533735690793728.0, 'train_loss': 0.4282976141921035, 'epoch': 12.0}



Final validation metrics: {'eval_loss': 1.7036932706832886, 'eval_token_accuracy': 0.8464462181679969, 'eval_span_precision_strict': 0.09523809523809523, 'eval_span_recall_strict': 0.1678832116788321, 'eval_span_f1_strict': 0.12153236459709377, 'eval_span_f1_3c': 0.12153236459709377, 'eval_runtime': 3.8891, 'eval_samples_per_second': 97.453, 'eval_steps_per_second': 6.171, 'epoch': 12.0}


In [9]:
def predict_dataset(ds: Dataset):
    out = trainer.predict(ds)
    preds = np.argmax(out.predictions, axis=-1)
    return ids_to_tags(out.label_ids, preds)


# --- Run on test split ---
true_tags_test, pred_tags_test = predict_dataset(dsd["test"])

print("=== seqeval (3C, strict span match) ===")
print(seq_classification_report(
    true_tags_test, pred_tags_test, mode="strict", digits=4, zero_division=0
))

# Headline numbers for chapter 4.5
strict_p = seq_precision(true_tags_test, pred_tags_test, mode="strict", zero_division=0)
strict_r = seq_recall(true_tags_test, pred_tags_test, mode="strict", zero_division=0)
strict_f = seq_f1(true_tags_test, pred_tags_test, mode="strict", zero_division=0)

print(f"\n=== Headline ===")
print(f"Span precision (strict): {strict_p:.4f}")
print(f"Span recall    (strict): {strict_r:.4f}")
print(f"Span F1        (strict): {strict_f:.4f}")

=== seqeval (3C, strict span match) ===
              precision    recall  f1-score   support

        DIST     0.1568    0.1941    0.1735       273

   micro avg     0.1568    0.1941    0.1735       273
   macro avg     0.1568    0.1941    0.1735       273
weighted avg     0.1568    0.1941    0.1735       273


=== Headline ===
Span precision (strict): 0.0998
Span recall    (strict): 0.1941
Span F1        (strict): 0.1318


In [10]:
EVAL_DIR = OUT_DIR / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

with open(EVAL_DIR / "summary_test.json", "w", encoding="utf-8") as f:
    json.dump({
        "model_name": MODEL_NAME,
        "scheme": "3C",
        "test_size": len(dsd["test"]),
        "span_precision_strict": strict_p,
        "span_recall_strict": strict_r,
        "span_f1_strict": strict_f,
    }, f, indent=2)

pred_records = []
for rec, pt in zip(dsd["test"], pred_tags_test):
    pred_records.append({
        "id": rec["id"],
        "is_distorted": rec["is_distorted"],
        "dominant_distortion": rec["dominant_distortion"],
        "n_spans": rec["n_spans"],
        "pred_tags": pt,
    })
pd.DataFrame(pred_records).to_parquet(EVAL_DIR / "predictions_test.parquet", index=False)
print(f"Saved evaluation outputs to: {EVAL_DIR}")

Saved evaluation outputs to: /content/drive/MyDrive/thesis_stage2/models/bio_3c_roberta_base/eval


In [13]:
import textwrap

def find_spans(tag_seq):
    """Same as runs_to_spans but returns (start, end) only — drops category."""
    spans = []
    cur = None
    for i, t in enumerate(tag_seq):
        if t == "O":
            if cur is not None:
                spans.append(cur); cur = None
        elif t.startswith("B-"):
            if cur is not None:
                spans.append(cur)
            cur = (i, i + 1)
        else:
            if cur is None:
                cur = (i, i + 1)
            else:
                cur = (cur[0], i + 1)
    if cur is not None:
        spans.append(cur)
    return spans


shown = 0
for idx in range(len(dsd["test"])):
    rec = dsd["test"][idx]
    if not rec["is_distorted"]:
        continue
    if shown >= 5:
        break

    out = trainer.predict(dsd["test"].select([idx]))
    preds = np.argmax(out.predictions, axis=-1)
    true_tags_record = ids_to_tags(out.label_ids, preds)[0][0]
    pred_tags_record = ids_to_tags(out.label_ids, preds)[1][0]

    gold_spans = find_spans(true_tags_record)
    pred_spans = find_spans(pred_tags_record)

    # Build a non-special-token list aligned to the filtered tag sequence
    decoded = tokenizer.convert_ids_to_tokens(rec["input_ids"])
    non_special = [tok for tok, lab in zip(decoded, rec["labels"]) if lab != -100]

    print(f"=== Record {idx} (id={rec['id']}, category={rec['dominant_distortion']}) ===")
    print(f"Gold spans (token positions): {gold_spans}")
    print(f"Pred spans (token positions): {pred_spans}\n")

    print("--- Gold span text ---")
    for sp in gold_spans:
        text = tokenizer.convert_tokens_to_string(non_special[sp[0]:sp[1]])
        print(f"  [{sp[0]}:{sp[1]}] {textwrap.shorten(text, width=200)}")

    print("--- Predicted span text ---")
    for sp in pred_spans:
        text = tokenizer.convert_tokens_to_string(non_special[sp[0]:sp[1]])
        print(f"  [{sp[0]}:{sp[1]}] {textwrap.shorten(text, width=200)}")
    print()
    shown += 1

=== Record 0 (id=179, category=Mind Reading) ===
Gold spans (token positions): [(14, 55), (383, 402)]
Pred spans (token positions): []

--- Gold span text ---
  [14:55] He is a gifted kid who doesn’t really like school. He speaks out of turn, has said mean things, and doesn’t really seem to care about authority or following the rules.
  [383:402] He gets really wound up and doesn’t listen and it feels like attention seeking behavior.
--- Predicted span text ---



=== Record 3 (id=791, category=All-or-nothing thinking) ===
Gold spans (token positions): [(121, 146)]
Pred spans (token positions): [(121, 136), (136, 137)]

--- Gold span text ---
  [121:146] I cant do anything because i dont want to stress my mother as well. therefore my sister can do what ever she wants.
--- Predicted span text ---
  [121:136] I cant do anything because i dont want to stress my mother as well.
  [136:137] therefore



=== Record 4 (id=874, category=Mind Reading) ===
Gold spans (token positions): [(318, 336)]
Pred spans (token positions): [(124, 146), (336, 356), (429, 453)]

--- Gold span text ---
  [318:336] I know the stuff the stalker said hurt him because he thought it was hurting me.
--- Predicted span text ---
  [124:146] I need this reassurance in order to feel better about myself and affirm that people do actually care about me.
  [336:356] I am alright with keeping the secret because I do not think he will understand why I did it.
  [429:453] I can’t lose him and I don’t think he would ever be able to get over this lie



=== Record 6 (id=2563, category=Overgeneralization) ===
Gold spans (token positions): [(80, 120)]
Pred spans (token positions): [(48, 79)]

--- Gold span text ---
  [80:120] Now I am at university my peers around me all have social lives and boyfriends and so it hurts more now. I tried making friends but seem to have a perpetual block for the last 10 years.
--- Predicted span text ---
  [48:79] When I was at school I cared but not as much about not having any friends because I was the highest performing so I saw it as necessary to my success



=== Record 7 (id=2414, category=Emotional Reasoning) ===
Gold spans (token positions): [(37, 52), (102, 121)]
Pred spans (token positions): [(37, 52)]

--- Gold span text ---
  [37:52] All my life, I have felt mostly ignored & criticized by my mother.
  [102:121] They also seemed to have learned much, but I always felt a little deprived for some reason.
--- Predicted span text ---
  [37:52] All my life, I have felt mostly ignored & criticized by my mother.



In [15]:
def overlap_match(pred_spans, gold_spans):
    tp = 0
    matched_gold = set()
    for ps in pred_spans:
        for gi, gs in enumerate(gold_spans):
            if gi in matched_gold: continue
            if ps[0] < gs[1] and ps[1] > gs[0]:  # any overlap
                tp += 1
                matched_gold.add(gi)
                break
    fp = len(pred_spans) - tp
    fn = len(gold_spans) - len(matched_gold)
    return tp, fp, fn

total_tp = total_fp = total_fn = 0
for idx in range(len(dsd["test"])):
    rec = dsd["test"][idx]
    out = trainer.predict(dsd["test"].select([idx]))
    preds = np.argmax(out.predictions, axis=-1)
    true_tags = ids_to_tags(out.label_ids, preds)[0][0]
    pred_tags = ids_to_tags(out.label_ids, preds)[1][0]
    gs = find_spans(true_tags)
    ps = find_spans(pred_tags)
    tp, fp, fn = overlap_match(ps, gs)
    total_tp += tp; total_fp += fp; total_fn += fn

p = total_tp / max(1, total_tp + total_fp)
r = total_tp / max(1, total_tp + total_fn)
f = 2*p*r/(p+r) if (p+r) else 0
print(f"Partial-overlap: P={p:.4f}  R={r:.4f}  F1={f:.4f}")

Partial-overlap: P=0.2566  R=0.4982  F1=0.3387
